# Multigrid for Poisson

Relaxation removes rough error in a few sweeps and smooth error hardly at all
(see [Poisson in 2D, and relaxation](poisson_relaxation.ipynb)). Multigrid
turns that weakness around. After a few sweeps the remaining error is smooth,
so it can be represented on a grid with half the points. There it looks
rougher, and relaxation works again. Recursing down a hierarchy of grids
gives a solver whose cost per digit doesn't grow with the problem, and the
best preconditioner there is for Poisson problems.

This notebook builds the hierarchy for −u″ = 1 on [0, 1], as the C++ example
does, and measures that claim. It holds only when the hierarchy grows with
the problem, which a fixed number of levels doesn't.

**Uses:** `mtl5.mg.multigrid_1d`, `mtl5.mg.make_restriction_1d`,
`mtl5.mg.make_prolongation_1d`, `mtl5.mg.galerkin`, `Multigrid.vcycle`,
`mtl5.sparse.cg`, `mtl5.generators.laplacian_1d`, and SciPy's `cg` with a
`LinearOperator`.

**From C++:** [`phase13_krylov_multigrid/multigrid_poisson.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase13_krylov_multigrid/multigrid_poisson.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse.linalg as spla

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)


def poisson_1d(n):
    # -u'' on n interior points of [0, 1]: MTL5's [-1, 2, -1] scaled by 1/h^2
    return ms.from_scipy(ms.to_scipy(mtl5.generators.laplacian_1d(n)) * (n + 1) ** 2)

## Moving between grids

A grid of n = 2ᵏ − 1 interior points coarsens to (n − 1)/2 points, every
other one. Two sparse matrices move vectors between the grids.

- **Restriction** R averages a fine-grid vector onto the coarse grid with the
  weights ¼, ½, ¼ (full weighting).
- **Prolongation** P interpolates a coarse vector linearly back to the fine
  grid.

They are transposes of each other up to a factor: R = ½ Pᵀ. The coarse-grid
operator is the Galerkin product R A P, and for this problem it is exactly
the Poisson matrix of the coarse grid.

In [ ]:
n = 63
A = poisson_1d(n)
R = mtl5.mg.make_restriction_1d(n)
P = mtl5.mg.make_prolongation_1d((n - 1) // 2)
R_np, P_np = ms.to_scipy(R).toarray(), ms.to_scipy(P).toarray()
print(f"R: {R.shape}, row 0 = {R_np[0, :4]};  P: {P.shape}, column 0 = {P_np[:4, 0]}")

coarse = ms.to_scipy(mtl5.mg.galerkin(R, A, P)).toarray()
assert np.allclose(R_np, 0.5 * P_np.T)
assert np.allclose(coarse, ms.to_scipy(poisson_1d((n - 1) // 2)).toarray())
print("R = P^T / 2, and R A P is the Poisson matrix on 31 points")

## A V-cycle

`multigrid_1d` builds the hierarchy from the fine matrix, coarsening by
Galerkin products. The C++ example uses three levels, 63 → 31 → 15, with two
Gauss–Seidel sweeps before and after each coarse-grid correction. A V-cycle
smooths, restricts the residual, recurses, interpolates the correction back
and smooths again. Each cycle multiplies the residual by roughly a constant
factor.

In [ ]:
b = np.ones(n)
A_np = ms.to_scipy(A)
mg = mtl5.mg.multigrid_1d(A, n_levels=3, smoother="gauss_seidel", nu_pre=2, nu_post=2)
print("levels:", mg.level_sizes)


def vcycle_history(mg, A_np, b, max_cycles=60, rtol=1e-10):
    x = mtl5.vector(np.zeros(len(b)))
    hist = [1.0]
    while hist[-1] > rtol and len(hist) <= max_cycles:
        x = mg.vcycle(x, mtl5.vector(b))
        hist.append(np.linalg.norm(b - A_np @ x.to_numpy()) / np.linalg.norm(b))
    return np.array(hist)


hist = vcycle_history(mg, A_np, b)
for cycle, r in enumerate(hist[1:], start=1):
    print(f"cycle {cycle}: relative residual {r:.2e}")
factors = hist[2:] / hist[1:-1]
print("reduction per cycle:", np.round(factors, 3))
assert hist[-1] <= 1e-10 and len(hist) - 1 <= 10

Seven cycles reach 1e-10, each one gaining more than a digit. CG takes 32
iterations on the same system.

## Does it scale?

The claim for multigrid is that the number of cycles doesn't depend on n. The
C++ example tests it only at n = 63. Repeating the solve on finer grids shows
that the claim needs a condition: the coarsest grid has to stay small.

- With the C++ example's **three levels**, the coarsest grid holds a quarter
  of the points, and it grows with n. `multigrid_1d` solves the coarsest
  problem with 200 sweeps of the smoother. That is plenty for 15 points and
  nowhere near enough for 255, where it leaves the smooth error that
  relaxation can't remove.
- With a **full hierarchy**, coarsening down to 3 points, the cycle count is
  the same at every size.

In [ ]:
ks = np.arange(6, 13)
sizes = 2**ks - 1
CAP = 60  # V-cycles before giving up
fixed, fixed_done, full, cg_its = [], [], [], []
for k, m in zip(ks, sizes):
    Am = poisson_1d(m)
    Am_np, bm = ms.to_scipy(Am), np.ones(m)
    h3 = vcycle_history(mtl5.mg.multigrid_1d(Am, n_levels=3), Am_np, bm, max_cycles=CAP)
    fixed.append(len(h3) - 1)
    fixed_done.append(h3[-1] <= 1e-10)
    deep = mtl5.mg.multigrid_1d(Am, n_levels=int(k) - 1)
    hf = vcycle_history(deep, Am_np, bm, max_cycles=CAP)
    assert hf[-1] <= 1e-10
    full.append(len(hf) - 1)
    _, info, report = ms.cg(Am, bm, rtol=1e-10, maxiter=10 * m, full_output=True)
    assert info == 0
    cg_its.append(report.iterations)
    three = f"{fixed[-1]:3d}" if fixed_done[-1] else f"not converged in {CAP}"
    print(
        f"n = {m:5d} ({len(deep.level_sizes)} levels, coarsest {deep.level_sizes[-1]}):  "
        f"V-cycles with 3 levels {three}, full hierarchy {full[-1]:2d};  CG {cg_its[-1]}"
    )

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.loglog(sizes, cg_its, "s-", color=SERIES[1], label="CG iterations")
done = np.array(fixed_done)
ax.loglog(
    sizes[done],
    np.array(fixed)[done],
    "^-",
    color=SERIES[2],
    label="V-cycles, 3 levels (C++ example)",
)
ax.loglog(
    sizes[~done],
    np.full((~done).sum(), CAP),
    "^",
    color=SERIES[2],
    markerfacecolor="none",
    label=f"3 levels: not converged in {CAP} cycles",
)
ax.loglog(sizes, full, "o-", color=SERIES[0], label="V-cycles, full hierarchy")
ax.set_xlabel("n")
ax.set_ylabel("to relative residual 1e-10")
ax.set_title("−u″ = 1: multigrid's count stays flat only with enough levels", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.grid(True, which="major")
plt.tight_layout()
plt.show()

full, fixed, cg_its = map(np.array, (full, fixed, cg_its))
assert full.max() - full.min() <= 2 and full.max() <= 12  # flat
assert fixed_done[0] and not any(fixed_done[3:])  # three levels: fine at 63, fail from 511 on
assert (cg_its[1:] / cg_its[:-1] > 1.8).all()  # CG grows like n

CG's iterations double with n, as its theory predicts for a condition number
that grows like n². The count is exactly (n + 1)/2 here, not n, because b = 1
is symmetric about the midpoint: only the symmetric half of the eigenvectors
appear in it, and CG ends after one step per eigenvalue it sees. Each V-cycle
costs a fixed number of sweeps per level, and the levels shrink geometrically,
so one cycle costs O(n). A flat cycle count makes the whole solve O(n).

## Multigrid as a preconditioner

One V-cycle from zero is an approximate inverse of A, and so a
preconditioner. Used inside CG it removes the iteration count's growth
entirely. CG needs a symmetric positive definite preconditioner, though, and
a V-cycle with forward Gauss–Seidel sweeps isn't symmetric: the forward sweep
transposed is a backward one. Symmetric Gauss–Seidel, a forward sweep followed
by a backward one, makes the V-cycle symmetric. The C++ example pairs CG with
the forward version, which converges in practice but outside CG's theory.

In [ ]:
def vcycle_operator(mg, m):
    # The V-cycle from x = 0 as a matrix: one column per unit vector.
    zero = mtl5.vector(np.zeros(m))
    return np.column_stack(
        [mg.vcycle(zero, mtl5.vector(np.eye(m)[i])).to_numpy() for i in range(m)]
    )


for smoother in ("gauss_seidel", "symmetric_gauss_seidel"):
    B = vcycle_operator(mtl5.mg.multigrid_1d(A, n_levels=5, smoother=smoother), n)
    asym = np.abs(B - B.T).max() / np.abs(B).max()
    print(f"{smoother:>23}: relative asymmetry of the V-cycle {asym:.1e}")
    if smoother == "symmetric_gauss_seidel":
        assert asym < 1e-12
        assert np.linalg.eigvalsh(B).min() > 0  # and positive definite
    else:
        assert asym > 1e-4

mtl5's own Krylov solvers take only mtl5's preconditioners, so the
V-cycle goes to SciPy's CG as a `LinearOperator`. That is the interop route,
and it applies the preconditioner from Python once per iteration. That's
fine here, because there are only a handful of iterations.

In [ ]:
pcg_its = []
for k, m in zip(ks, sizes):
    Am = poisson_1d(m)
    Am_np, bm = ms.to_scipy(Am), np.ones(m)
    mg_m = mtl5.mg.multigrid_1d(Am, n_levels=int(k) - 1, smoother="symmetric_gauss_seidel")

    def apply(r, mg_m=mg_m, m=m):
        rr = np.ascontiguousarray(np.ravel(r), dtype=np.float64)
        return mg_m.vcycle(mtl5.vector(np.zeros(m)), mtl5.vector(rr)).to_numpy()

    M = spla.LinearOperator((m, m), matvec=apply, dtype=np.float64)
    count = [0]
    x, info = spla.cg(
        Am_np,
        bm,
        M=M,
        rtol=1e-10,
        maxiter=100,
        callback=lambda xk: count.__setitem__(0, count[0] + 1),
    )
    assert info == 0
    rel = np.linalg.norm(bm - Am_np @ x) / np.linalg.norm(bm)
    pcg_its.append(count[0])
    alone = cg_its[k - 6]
    print(f"n = {m:5d}: CG + V-cycle {count[0]} iterations (residual {rel:.1e}), CG alone {alone}")

pcg_its = np.array(pcg_its)
assert pcg_its.max() - pcg_its.min() <= 1 and pcg_its.max() <= 8

Five iterations at every size, from 63 unknowns to 4095, against 2048 for CG
alone at the largest.

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `itl::mg::make_restriction_1d(n)`, `make_prolongation_1d(n)` | `mtl5.mg.make_restriction_1d(n)`, `mtl5.mg.make_prolongation_1d(n)` |
| the levels' matrices built one by one with `make_poisson` | `mtl5.mg.multigrid_1d(A, n_levels=...)`, Galerkin coarsening; `mtl5.mg.galerkin(R, A, P)` |
| `itl::mg::multigrid<double> mg(levels, R, P, smoother, coarse, 2, 2)` | `multigrid_1d(A, smoother=..., nu_pre=2, nu_post=2)` |
| `mg.vcycle(x, b)` | `x = mg.vcycle(x, b)`, returning the updated x |
| `itl::cg(A, x, b, mg, iter)` | SciPy's `cg` with `M=LinearOperator(... mg.vcycle ...)` |

Two changes from the C++ program, both measured above. The hierarchy grows
with n, because three fixed levels stop scaling. And the preconditioner for
CG uses the symmetric Gauss–Seidel smoother, because CG's theory needs a
symmetric preconditioner.